# Daily Synthetic In-Store Song Play Telemetry Generator

Generates one store-local trading day of synthetic play events (default: yesterday in Australia/Sydney, or the `run_date` job parameter) as JSON Lines in `/Volumes/cricket_australia_sponsor/song_telemetry/events/landing/batch_<YYYYMMDD>/`. Reference data (stores, zones, devices, catalogue) is fixed-seed so dimension keys stay stable across runs; events are seeded per day so re-runs are idempotent.

In [0]:
import os, json, uuid
from datetime import datetime, date, timedelta, timezone
from zoneinfo import ZoneInfo
import numpy as np

# ---- Parameters (job parameters or widgets) ----
# run_date: YYYY-MM-DD store-local trading day to generate; blank = yesterday in Australia/Sydney
dbutils.widgets.text("run_date", "")
dbutils.widgets.text("n_events", "10000")
_run_date_str = dbutils.widgets.get("run_date").strip()
RUN_DATE = (date.fromisoformat(_run_date_str) if _run_date_str
            else datetime.now(ZoneInfo("Australia/Sydney")).date() - timedelta(days=1))
N_EVENTS = int(dbutils.widgets.get("n_events"))

# ---- Reference data: fixed seed so store/device/content keys are stable across runs ----
SEED = 42
REFERENCE_START_DATE = date(2026, 11, 12)  # anchors device firmware-upgrade dates
REFERENCE_N_DAYS = 14
N_STORES = 40
N_ARTISTS = 220
N_MUSIC_TRACKS = 800

# ---- Events: one local trading day, seeded per day so re-runs are idempotent ----
START_DATE = RUN_DATE
N_DAYS = 1
EVENT_SEED = SEED * 100_000 + RUN_DATE.toordinal()

VOLUME_PATH = "/Volumes/cricket_australia_sponsor/song_telemetry/events"
BATCH_ID = f"batch_{RUN_DATE:%Y%m%d}"
OUTPUT_DIR = f"{VOLUME_PATH}/landing/{BATCH_ID}"

# All randomness goes through one numpy Generator (PCG64) so output is reproducible on any runtime
rng = np.random.default_rng(SEED)
UTC = timezone.utc

def pick(seq):
    """Uniform choice from a sequence using the current seeded rng."""
    return seq[int(rng.integers(len(seq)))]

def new_uuid() -> str:
    """Deterministic UUID4 driven by the current seeded rng."""
    return str(uuid.UUID(bytes=rng.bytes(16), version=4))

print(f"run_date={RUN_DATE}, n_events={N_EVENTS}, output={OUTPUT_DIR}")

In [0]:
# city, state, timezone, lat, lon, weight
CITIES = [
    ("Sydney", "NSW", "Australia/Sydney", -33.8688, 151.2093, 0.26),
    ("Melbourne", "VIC", "Australia/Melbourne", -37.8136, 144.9631, 0.24),
    ("Brisbane", "QLD", "Australia/Brisbane", -27.4698, 153.0251, 0.16),
    ("Perth", "WA", "Australia/Perth", -31.9505, 115.8605, 0.12),
    ("Adelaide", "SA", "Australia/Adelaide", -34.9285, 138.6007, 0.09),
    ("Gold Coast", "QLD", "Australia/Brisbane", -28.0167, 153.4000, 0.05),
    ("Canberra", "ACT", "Australia/Sydney", -35.2809, 149.1300, 0.04),
    ("Hobart", "TAS", "Australia/Hobart", -42.8821, 147.3272, 0.02),
    ("Darwin", "NT", "Australia/Darwin", -12.4634, 130.8456, 0.02),
]
RETAILERS = [
    ("RTL001", "Coastline Apparel", "fashion"),
    ("RTL002", "Urban Pantry", "grocery"),
    ("RTL003", "Velocity Sports", "sporting_goods"),
    ("RTL004", "Hearth & Home", "homewares"),
]
# format: (open_hour, close_hour, selection weight, event volume weight)
STORE_FORMATS = {
    "flagship": (9, 21, 0.15, 2.0),
    "shopping_centre": (9, 21, 0.45, 1.3),
    "high_street": (9, 18, 0.30, 1.0),
    "outlet": (10, 18, 0.10, 0.7),
}
ZONE_BASE_VOLUME = {"Main Floor": 55, "Fitting Rooms": 45, "Checkout": 50, "Cafe": 40, "Entrance": 60}
PLAYER_MODELS = ["QPlayer-200", "QPlayer-300", "QPlayer-Mini"]
NETWORK_TYPES, NETWORK_W = ["ethernet", "wifi", "4g"], [0.55, 0.35, 0.10]
FIRMWARE_INITIAL, FIRMWARE_UPGRADE = "4.2.1", "4.3.0"

city_w = np.array([c[5] for c in CITIES]); city_w /= city_w.sum()
fmt_names = list(STORE_FORMATS)
fmt_w = np.array([STORE_FORMATS[f][2] for f in fmt_names]); fmt_w /= fmt_w.sum()

stores, zones_by_store, device_by_zone = [], {}, {}
zone_seq = device_seq = 0
for i in range(1, N_STORES + 1):
    city, state, tz, lat, lon, _ = CITIES[rng.choice(len(CITIES), p=city_w)]
    rid, rname, rcat = RETAILERS[rng.integers(len(RETAILERS))]
    fmt = fmt_names[rng.choice(len(fmt_names), p=fmt_w)]
    open_h, close_h, _, vol_w = STORE_FORMATS[fmt]
    store = dict(
        store_id=f"STR{i:04d}", store_name=f"{rname} {city} #{i}",
        retailer_id=rid, retailer_name=rname, retailer_category=rcat,
        store_format=fmt, city=city, state=state, country="AU", store_timezone=tz,
        latitude=round(lat + rng.uniform(-0.08, 0.08), 5),
        longitude=round(lon + rng.uniform(-0.08, 0.08), 5),
        open_hour=open_h, close_hour=close_h, event_weight=vol_w,
    )
    stores.append(store)

    # Zones: Main Floor always, plus extras (flagships get the most)
    n_extra = 2 if fmt == "flagship" else int(rng.integers(0, 2))
    zone_names = ["Main Floor"] + list(rng.choice(list(ZONE_BASE_VOLUME)[1:], n_extra, replace=False))
    zones_by_store[store["store_id"]] = []
    for zn in zone_names:
        zone_seq += 1; device_seq += 1
        zone = dict(zone_id=f"ZON{zone_seq:05d}", zone_name=str(zn), base_volume=ZONE_BASE_VOLUME[zn])
        zones_by_store[store["store_id"]].append(zone)
        # ~50% of devices get a firmware upgrade on a fixed date (drives SCD2 on dim_device)
        upgrade_date = (REFERENCE_START_DATE + timedelta(days=int(rng.integers(2, REFERENCE_N_DAYS - 2)))
                        if rng.random() < 0.5 else None)
        device_by_zone[zone["zone_id"]] = dict(
            device_id=f"DEV{device_seq:05d}",
            player_model=PLAYER_MODELS[rng.integers(len(PLAYER_MODELS))],
            network_type=str(rng.choice(NETWORK_TYPES, p=NETWORK_W)),
            upgrade_date=upgrade_date,
        )

print(f"stores={len(stores)}, zones={zone_seq}, devices={device_seq}")

In [0]:
GENRES = {  # genre -> bpm range
    "Pop": (95, 130), "Indie": (90, 140), "Electronic": (115, 128), "R&B": (70, 100), "Rock": (100, 150),
    "Acoustic": (70, 110), "Jazz": (80, 140), "Hip-Hop": (80, 100), "Chill": (70, 100), "Country": (80, 120),
}
LABELS = {"Southern Cross Records": "SCR", "Harbour Lights Music": "HLM", "Red Dirt Recordings": "RDR",
          "Outback Sound Co": "OSC", "Tasman Audio": "TAS"}
ADJ = ["Velvet", "Neon", "Golden", "Silver", "Midnight", "Coastal", "Electric", "Wild", "Paper", "Crystal",
       "Southern", "Hollow", "Lunar", "Copper", "Indigo", "Rapid", "Quiet", "Dusty", "Static", "Amber"]
NOUN = ["Kites", "Harbour", "Echoes", "Pines", "Rivers", "Foxes", "Lanterns", "Waves", "Atlas", "Comets",
        "Horizon", "Mirrors", "Saints", "Tides", "Wolves", "Gardens", "Signals", "Embers", "Sparrows", "Dunes"]
TW1 = ["Summer", "Broken", "Endless", "Golden", "Lost", "Midnight", "Sweet", "Neon", "Falling", "Running",
       "Paper", "Silent", "Burning", "City", "Ocean", "Electric", "Little", "Wild", "Northern", "Blue"]
TW2 = ["Lights", "Hearts", "Roads", "Dreams", "Skies", "Nights", "Signals", "Shadows", "Love", "Fire",
       "Rain", "Days", "Waves", "Lines", "Stars", "Echoes", "Motion", "Gold", "Fever", "Home"]
genre_names, label_names = list(GENRES), list(LABELS)

# Artists (unique names), each with a home genre, label and 1-4 albums
name_pool = [f"{a} {n}" for a in ADJ for n in NOUN]
name_pool = [name_pool[k] for k in rng.permutation(len(name_pool))]
artists = []
for i in range(1, N_ARTISTS + 1):
    nm = name_pool[i - 1]
    artists.append(dict(
        artist_id=f"ART{i:04d}", artist_name=("The " + nm) if rng.random() < 0.4 else nm,
        genre=genre_names[rng.integers(len(genre_names))], label=label_names[rng.integers(len(label_names))],
        albums=[f"{pick(TW1)} {pick(TW2)}" for _ in range(int(rng.integers(1, 5)))],
    ))

EMPTY_MUSIC = dict(isrc=None, artist_id=None, artist_name=None, album_name=None, label=None, genre=None,
                   release_year=None, bpm=None, energy=None, explicit_flag=False)

# Music tracks
tracks = []
for i in range(1, N_MUSIC_TRACKS + 1):
    a = artists[rng.integers(len(artists))]
    genre = a["genre"] if rng.random() < 0.8 else genre_names[rng.integers(len(genre_names))]
    lo, hi = GENRES[genre]
    bpm = int(rng.integers(lo, hi + 1))
    year = int(np.clip(2026 - rng.exponential(7), 1970, 2026))
    tracks.append(dict(
        content_id=f"TRK{i:05d}", content_type="music", title=f"{pick(TW1)} {pick(TW2)}",
        isrc=f"AU{LABELS[a['label']]}{year % 100:02d}{i:05d}",
        artist_id=a["artist_id"], artist_name=a["artist_name"], album_name=pick(a["albums"]),
        label=a["label"], genre=genre, release_year=year, bpm=bpm,
        energy=round(float(np.clip((bpm - 60) / 100 + rng.normal(0, 0.12), 0.05, 0.99)), 2),
        explicit_flag=bool(rng.random() < (0.35 if genre == "Hip-Hop" else 0.05)),
        content_duration_ms=int(np.clip(rng.normal(215, 40), 120, 420) * 1000),
    ))
# Long-tail (Zipf-like) popularity
popularity = 1.0 / (rng.permutation(N_MUSIC_TRACKS) + 1) ** 1.07

# Retailer-specific ads and generic announcements
AD_TOPICS = ["Summer Sale", "New Arrivals", "Members Rewards", "Click & Collect", "Weekend Deals", "Gift Cards"]
ads_by_retailer, ad_seq = {}, 0
for rid, rname, _ in RETAILERS:
    ads_by_retailer[rid] = []
    for t in AD_TOPICS:
        ad_seq += 1
        ads_by_retailer[rid].append(dict(EMPTY_MUSIC, content_id=f"AD{ad_seq:04d}", content_type="ad",
                                         title=f"{rname} - {t}", content_duration_ms=int(rng.choice([15000, 30000]))))
ANN_TOPICS = ["Welcome Message", "Store Closing Reminder", "Staff Call to Front", "Safety Announcement",
              "Loyalty Program Reminder", "Lost Child Protocol"]
announcements = [dict(EMPTY_MUSIC, content_id=f"ANN{j:03d}", content_type="announcement", title=t,
                      content_duration_ms=int(rng.integers(20, 46)) * 1000)
                 for j, t in enumerate(ANN_TOPICS, 1)]

# Playlists scheduled by daypart
PLAYLISTS = [
    ("PL001", "Morning Coffee", "morning", ["Acoustic", "Jazz", "Chill", "Indie"]),
    ("PL002", "Fresh Start", "morning", ["Pop", "Indie", "Acoustic"]),
    ("PL003", "Midday Mix", "midday", ["Pop", "R&B", "Electronic"]),
    ("PL004", "Lunch Rush", "midday", ["Pop", "Rock", "Hip-Hop"]),
    ("PL005", "Afternoon Groove", "afternoon", ["R&B", "Electronic", "Indie"]),
    ("PL006", "Retail Energy", "afternoon", ["Pop", "Electronic", "Hip-Hop", "Rock"]),
    ("PL007", "Evening Wind Down", "evening", ["Chill", "Jazz", "Acoustic", "Country"]),
    ("PL008", "Late Trade", "evening", ["Pop", "Indie", "R&B"]),
]
playlists_by_daypart = {}
for pid, pname, dp, genres in PLAYLISTS:
    idx = np.array([k for k, t in enumerate(tracks) if t["genre"] in genres])
    p = popularity[idx] / popularity[idx].sum()
    playlists_by_daypart.setdefault(dp, []).append(dict(playlist_id=pid, playlist_name=pname, daypart=dp, track_idx=idx, track_p=p))

print(f"artists={len(artists)}, tracks={len(tracks)}, ads={ad_seq}, announcements={len(announcements)}, playlists={len(PLAYLISTS)}")

In [0]:
# Re-seed for events so each run_date produces different (but reproducible) plays
rng = np.random.default_rng(EVENT_SEED)

def daypart(hour: int) -> str:
    return "morning" if hour < 11 else "midday" if hour < 14 else "afternoon" if hour < 17 else "evening"

def iso_utc(dt: datetime) -> str:
    return dt.astimezone(UTC).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

END_REASONS = ["completed", "skipped", "interrupted", "override", "power_loss"]
END_P = {"music": [0.88, 0.06, 0.03, 0.02, 0.01], "music_4g": [0.82, 0.06, 0.09, 0.02, 0.01],
         "non_music": [0.97, 0.0, 0.02, 0.0, 0.01]}
BUFFER_RATE = {"ethernet": 0.05, "wifi": 0.3, "4g": 0.9}
CACHE_RATE = {"ethernet": 0.01, "wifi": 0.04, "4g": 0.12}
DAYPART_VOL = {"morning": -5, "midday": 5, "afternoon": 3, "evening": -3}

# Weekend days and bigger formats get more sessions
day_w = np.array([1.35 if (START_DATE + timedelta(d)).weekday() >= 5 else 1.0 for d in range(N_DAYS)]); day_w /= day_w.sum()
store_w = np.array([s["event_weight"] for s in stores]); store_w /= store_w.sum()

events = []
while len(events) < N_EVENTS:
    # A "session" = a contiguous run of plays on one zone's player
    s = stores[rng.choice(N_STORES, p=store_w)]
    d = START_DATE + timedelta(days=int(rng.choice(N_DAYS, p=day_w)))
    zone = pick(zones_by_store[s["store_id"]])
    dev = device_by_zone[zone["zone_id"]]
    tz = ZoneInfo(s["store_timezone"])
    day0 = datetime(d.year, d.month, d.day)
    local_t = day0 + timedelta(minutes=int(rng.integers(s["open_hour"] * 60, s["close_hour"] * 60 - 30)),
                               seconds=int(rng.integers(60)))
    close_t = day0 + timedelta(hours=s["close_hour"])
    firmware = FIRMWARE_UPGRADE if dev["upgrade_date"] and d >= dev["upgrade_date"] else FIRMWARE_INITIAL
    net = dev["network_type"]
    playlist, since_ad = None, 0

    for _ in range(int(rng.integers(10, 45))):
        if local_t >= close_t or len(events) >= N_EVENTS:
            break
        dp = daypart(local_t.hour)
        if playlist is None or playlist["daypart"] != dp:
            playlist = pick(playlists_by_daypart[dp])

        r = rng.random()
        if since_ad >= 5 and r < 0.25:
            item, since_ad = pick(ads_by_retailer[s["retailer_id"]]), 0
        elif r > 0.98:
            item = pick(announcements)
        else:
            item = tracks[int(rng.choice(playlist["track_idx"], p=playlist["track_p"]))]
            since_ad += 1

        p_key = "non_music" if item["content_type"] != "music" else ("music_4g" if net == "4g" else "music")
        end_reason = str(rng.choice(END_REASONS, p=END_P[p_key]))
        dur = item["content_duration_ms"]
        played = dur if end_reason == "completed" else int(dur * rng.uniform(0.05, 0.95))
        buffer_events = int(rng.poisson(BUFFER_RATE[net])) + (int(rng.integers(1, 4)) if end_reason == "interrupted" else 0)
        from_cache = bool(rng.random() < CACHE_RATE[net])
        local_aware = local_t.replace(tzinfo=tz)
        # Cached (offline) plays upload late: 1h-48h; otherwise a few seconds
        delay_s = float(rng.uniform(3600, 172800)) if from_cache else float(rng.lognormal(1.5, 0.8))
        ingest = local_aware + timedelta(milliseconds=played, seconds=delay_s)

        events.append(dict(
            event_id=new_uuid(),
            event_ts_utc=iso_utc(local_aware),
            event_ts_local=local_aware.isoformat(timespec="milliseconds"),
            ingest_ts_utc=iso_utc(ingest),
            # store / location
            store_id=s["store_id"], store_name=s["store_name"], retailer_id=s["retailer_id"],
            retailer_name=s["retailer_name"], retailer_category=s["retailer_category"], store_format=s["store_format"],
            city=s["city"], state=s["state"], country=s["country"], store_timezone=s["store_timezone"],
            latitude=s["latitude"], longitude=s["longitude"],
            zone_id=zone["zone_id"], zone_name=zone["zone_name"],
            # device
            device_id=dev["device_id"], player_model=dev["player_model"], firmware_version=firmware, network_type=net,
            # content
            content_id=item["content_id"], content_type=item["content_type"], title=item["title"], isrc=item["isrc"],
            artist_id=item["artist_id"], artist_name=item["artist_name"], album_name=item["album_name"],
            label=item["label"], genre=item["genre"], release_year=item["release_year"], bpm=item["bpm"],
            energy=item["energy"], explicit_flag=item["explicit_flag"], content_duration_ms=dur,
            # schedule
            playlist_id=playlist["playlist_id"], playlist_name=playlist["playlist_name"], daypart=dp,
            # playback measures
            played_duration_ms=played, completion_pct=round(100.0 * played / dur, 1), end_reason=end_reason,
            volume_level=int(np.clip(zone["base_volume"] + DAYPART_VOL[dp] + rng.normal(0, 4), 0, 100)),
            buffer_events=buffer_events, played_from_cache=from_cache,
            # licensing
            rights_org="APRA AMCOS / PPCA" if item["content_type"] == "music" else None,
            royalty_reportable=bool(item["content_type"] == "music" and played >= 30000),
        ))
        local_t += timedelta(milliseconds=played, seconds=float(rng.uniform(0.5, 3.0)))

print(f"generated {len(events)} events for {RUN_DATE}")

In [0]:
# One JSON Lines file per local event date in this run's batch folder
os.makedirs(OUTPUT_DIR, exist_ok=True)
by_date = {}
for e in events:
    by_date.setdefault(e["event_ts_local"][:10], []).append(e)

def _np_default(o):
    if isinstance(o, np.generic):
        return o.item()
    raise TypeError(f"Not JSON serialisable: {type(o)}")

for dt_str, rows in sorted(by_date.items()):
    path = f"{OUTPUT_DIR}/song_play_events_{dt_str}.json"
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row, default=_np_default) + "\n")
    print(f"{path}: {len(rows)} rows")

In [0]:
# Read back only the files written by this run (the batch folder may hold earlier files)
written_paths = [f"{OUTPUT_DIR}/song_play_events_{d}.json" for d in sorted(by_date)]
raw = spark.read.json(written_paths)
n_rows = raw.count()
n_ids = raw.select("event_id").distinct().count()
print(f"rows: {n_rows} | distinct event_id: {n_ids} | columns: {len(raw.columns)}")
assert n_rows == N_EVENTS and n_ids == N_EVENTS, "Unexpected row count or duplicate event_ids"